# MP6 - Pressly Ad Agency

**Userid:** qpn8pr (all tables and views use extended names).

**Normalization summary (3NF)**
- Key of the big table: (AdCampaignID, TypeID)
- 1NF: all values are atomic, no repeating groups.
- 2NF: Name, Start, Duration depend only on AdCampaignID -> `AD_CAMPAIGN`. Media, Range, ManagerID depend only on TypeID -> `AD_TYPE`. Budget Pctg depends on the full key -> `CAMPAIGN_TYPE`.
- 3NF: Manager Name depends on ManagerID (not on the key) -> `MANAGER`.

## Step 7 - Drop the view and tables if they exist
Order matters: the view first, then the tables holding foreign keys, then the tables they reference.

In [22]:
DROP VIEW IF EXISTS qpn8pr.BIG_TABLE;
DROP TABLE IF EXISTS qpn8pr.CAMPAIGN_TYPE;
DROP TABLE IF EXISTS qpn8pr.AD_TYPE;
DROP TABLE IF EXISTS qpn8pr.MANAGER;
DROP TABLE IF EXISTS qpn8pr.AD_CAMPAIGN;

Commands completed successfully.

Total execution time: 00:00:00.022

## Step 2 - Create the tables
MANAGER: one row per manager. PK = ManagerID.

In [23]:
CREATE TABLE qpn8pr.MANAGER (
    ManagerID   VARCHAR(10) NOT NULL PRIMARY KEY,
    ManagerName VARCHAR(50) NOT NULL
);

Commands completed successfully.

Total execution time: 00:00:00.032

AD_TYPE: one row per type of ad (media + range), each managed by one manager. PK = TypeID, FK = ManagerID.

In [24]:
CREATE TABLE qpn8pr.AD_TYPE (
    TypeID    INT         NOT NULL PRIMARY KEY,
    Media     VARCHAR(20) NOT NULL,
    [Range]   VARCHAR(20) NOT NULL,
    ManagerID VARCHAR(10) NOT NULL,
    FOREIGN KEY (ManagerID) REFERENCES qpn8pr.MANAGER(ManagerID)
);

Commands completed successfully.

Total execution time: 00:00:00.023

AD_CAMPAIGN: one row per campaign. PK = AdCampaignID. Start is stored as a DATE and Duration (days) as an INT.

In [25]:
CREATE TABLE qpn8pr.AD_CAMPAIGN (
    AdCampaignID INT         NOT NULL PRIMARY KEY,
    Name         VARCHAR(50) NOT NULL,
    StartDate    DATE        NOT NULL,
    Duration     INT         NOT NULL
);

Commands completed successfully.

Total execution time: 00:00:00.020

CAMPAIGN_TYPE: association table (many-to-many between campaigns and ad types). Composite PK (AdCampaignID, TypeID), two FKs. BudgetPct is stored as a decimal (0.50 = 50%) and formatted with a % sign only in the view.

In [26]:
CREATE TABLE qpn8pr.CAMPAIGN_TYPE (
    AdCampaignID INT           NOT NULL,
    TypeID       INT           NOT NULL,
    BudgetPct    DECIMAL(3,2)  NOT NULL,
    PRIMARY KEY (AdCampaignID, TypeID),
    FOREIGN KEY (AdCampaignID) REFERENCES qpn8pr.AD_CAMPAIGN(AdCampaignID),
    FOREIGN KEY (TypeID)       REFERENCES qpn8pr.AD_TYPE(TypeID)
);

Commands completed successfully.

Total execution time: 00:00:00.021

## Step 3 - Populate the tables
Parents first (MANAGER, AD_CAMPAIGN), then AD_TYPE, then the association table. Note m202 is inserted as 'Elizabeth' and renamed in Step 4.

In [27]:
INSERT INTO qpn8pr.MANAGER (ManagerID, ManagerName) VALUES
    ('m200', 'Emma'),
    ('m201', 'Kate'),
    ('m202', 'Elizabeth'),
    ('m203', 'Nadia'),
    ('m204', 'Morgan'),
    ('m205', 'Bella'),
    ('m209', 'Luci');

(7 rows affected)

Total execution time: 00:00:00.020

In [28]:
INSERT INTO qpn8pr.AD_CAMPAIGN (AdCampaignID, Name, StartDate, Duration) VALUES
    (111, 'SummerFun',  '2025-06-06', 14),
    (222, 'SummerZing', '2025-06-08', 28),
    (333, 'FunSpring',  '2025-06-09', 28),
    (444, 'FallStyle',  '2025-06-09', 7),
    (555, 'FallColors', '2025-06-09', 14),
    (666, 'WinterCharm','2025-06-06', 7);

(6 rows affected)

Total execution time: 00:00:00.023

In [29]:
INSERT INTO qpn8pr.AD_TYPE (TypeID, Media, [Range], ManagerID) VALUES
    (1, 'TV',         'Local',    'm200'),
    (2, 'TV',         'National', 'm204'),
    (3, 'Social',     'Local',    'm205'),
    (4, 'Social',     'National', 'm209'),
    (5, 'Web',        'Local',    'm203'),
    (6, 'Web',        'National', 'm201'),
    (7, 'Influencer', 'Local',    'm202');

(7 rows affected)

Total execution time: 00:00:00.031

In [30]:
INSERT INTO qpn8pr.CAMPAIGN_TYPE (AdCampaignID, TypeID, BudgetPct) VALUES
    (111, 1, 0.50),
    (111, 2, 0.50),
    (222, 1, 0.60),
    (222, 3, 0.30),
    (222, 5, 0.10),
    (333, 3, 0.80),
    (333, 4, 0.20),
    (444, 6, 1.00),
    (555, 3, 1.00),
    (666, 7, 1.00);

(10 rows affected)

Total execution time: 00:00:00.028

## Step 4 - Rename manager m202 to 'Bethel'

In [31]:
UPDATE qpn8pr.MANAGER
SET ManagerName = 'Bethel'
WHERE ManagerID = 'm202';

(1 row affected)

Total execution time: 00:00:00.020

## Step 5 - Create the BIG_TABLE view
Joins the four tables and reproduces the original spreadsheet: same column names, order, and formats (date as m/d/yyyy, budget as a percentage with %). `TOP 100 PERCENT ... ORDER BY` keeps the row order inside the view.

In [32]:
CREATE VIEW qpn8pr.BIG_TABLE AS
SELECT TOP 100 PERCENT
    c.AdCampaignID                                   AS AdCampaignID,
    c.Name                                           AS [Name],
    FORMAT(c.StartDate, 'M/d/yyyy', 'en-US')         AS [Start],
    c.Duration                                       AS Duration,
    t.TypeID                                         AS TypeID,
    t.Media                                          AS Media,
    t.[Range]                                        AS [Range],
    FORMAT(ct.BudgetPct * 100, '0') + '%'            AS [Budget Pctg],
    m.ManagerID                                      AS ManagerID,
    m.ManagerName                                    AS [Manager Name]
FROM qpn8pr.AD_CAMPAIGN c
JOIN qpn8pr.CAMPAIGN_TYPE ct ON c.AdCampaignID = ct.AdCampaignID
JOIN qpn8pr.AD_TYPE t        ON ct.TypeID = t.TypeID
JOIN qpn8pr.MANAGER m        ON t.ManagerID = m.ManagerID
ORDER BY c.AdCampaignID, t.TypeID;

Commands completed successfully.

Total execution time: 00:00:00.025

## Step 6 - Show the view

In [33]:
SELECT * FROM qpn8pr.BIG_TABLE
ORDER BY AdCampaignID, TypeID;

(10 rows affected)

AdCampaignID | Name        | Start    | Duration | TypeID | Media      | Range    | Budget Pctg | ManagerID | Manager Name
-------------+-------------+----------+----------+--------+------------+----------+-------------+-----------+-------------
111          | SummerFun   | 6/6/2025 | 14       | 1      | TV         | Local    | 50%         | m200      | Emma        
111          | SummerFun   | 6/6/2025 | 14       | 2      | TV         | National | 50%         | m204      | Morgan      
222          | SummerZing  | 6/8/2025 | 28       | 1      | TV         | Local    | 60%         | m200      | Emma        
222          | SummerZing  | 6/8/2025 | 28       | 3      | Social     | Local    | 30%         | m205      | Bella       
222          | SummerZing  | 6/8/2025 | 28       | 5      | Web        | Local    | 10%         | m203      | Nadia       
333          | FunSpring   | 6/9/2025 | 28       | 3      | Social     | Local    | 80%         | m205      | Bella    